In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import absolute_import, division, print_function
import torch
from trainer_endoda3 import Trainer
from options_endoda3 import MonodepthOptions


/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [6]:
# Minimal options for testing
options = MonodepthOptions()
args = [
    '--batch_size', '2',
    '--num_workers', '1',
    '--of_samples',
    '--of_samples_num', '10',
    '--frame_ids', '0', '-1', '1',
    '--dataset', 'endovis',
    '--data_path', '/mnt/cluster/workspaces/jinjingxu/SCARED_Images_Resized/',
    '--log_dir', '/tmp/endoda_debug',
    '--compute_metrics',
    '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-depth-wowrapper.yaml'
    # '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-all-wowrapper.yaml',
]
opts = options.parse_notebook(args)

# Initialize trainer
trainer = Trainer(opts)
print(f"Trainer initialized on {trainer.device}")


Loading depth model setting from config: /mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-depth-wowrapper.yaml
[INFO ] using MLP layer as FFN

Loading pretrained weights from depth-anything/da3-base
[INFO ] using MLP layer as FFN

depth_model state dict info:
  Missing keys: 48
    Key prefixes (up to 3 levels): ['backbone.blocks.0', 'backbone.blocks.1', 'backbone.blocks.10', 'backbone.blocks.11', 'backbone.blocks.2', 'backbone.blocks.3', 'backbone.blocks.4', 'backbone.blocks.5', 'backbone.blocks.6', 'backbone.blocks.7', 'backbone.blocks.8', 'backbone.blocks.9']
  Unexpected keys: 74
    Key prefixes (up to 3 levels): ['cam_dec.backbone.0', 'cam_dec.backbone.2', 'cam_dec.fc_fov.0', 'cam_dec.fc_qvec.bias', 'cam_dec.fc_qvec.weight', 'cam_dec.fc_t.bias', 'cam_dec.fc_t.weight', 'cam_enc.pose_branch.fc1', 'cam_enc.pose_branch.fc2', 'cam_enc.token_norm.bias', 'cam_enc.token_norm.weight', 'cam_enc.trunk.0', 'cam_enc.trunk.1', 'cam_enc.trunk.2', 'cam_enc.trunk.3'

In [ ]:
# Get sample batch
trainer.step = 0
trainer.set_train()
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)

# Forward pass
outputs, losses = trainer.process_batch(inputs)

print("Forward pass completed!")
print(f"Output keys: {len(outputs)} keys")
print(f"Loss: {losses['loss'].item():.6f}")
print(f"Loss components: {list(losses.keys())}")


inputs shape:  torch.Size([2, 3, 256, 320])
Singel-frame input:  True
x_mv shape:  torch.Size([2, 1, 3, 256, 320])
Resizing input from 256x320 to 224x280
depth shape: torch.Size([2, 1, 224, 280])
conf shape: torch.Size([2, 1, 224, 280])
extrinsics shape: {}
intrinsics shape: {}


/mnt/cluster/environments/jinjingxu/pkg/envs/cu12/lib/python3.11/site-packages/torch/nn/functional.py:4902: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warnings.warn(


Forward pass completed!
Output keys: 128 keys
Loss: 0.160074
Loss components: ['loss/0', 'loss/1', 'loss/2', 'loss/3', 'loss']


: 

## 2. Test Loss Computation (compute_losses)


In [4]:
# Compute losses explicitly
losses = trainer.compute_losses(inputs, outputs)

print("Loss computation:")
for key, val in losses.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")


Loss computation:
  loss/0: 0.151731
  loss/1: 0.183570
  loss/2: 0.205991
  loss/3: 0.166883
  loss: 0.177044


## 3. Test Metrics Computation


In [5]:
# Get validation batch (has GT depth and poses)
trainer.set_eval()
val_iter = iter(trainer.val_loader)
val_inputs = next(val_iter)

# Forward pass
with torch.no_grad():
    val_outputs, val_losses = trainer.process_batch(val_inputs)

# Compute depth metrics
from utils.metrics import compute_depth_metrics, compute_pose_metrics

depth_metrics = compute_depth_metrics(val_inputs, val_outputs)
pose_metrics = compute_pose_metrics(val_inputs, val_outputs, opts.frame_ids)

print("Depth Metrics:")
if depth_metrics:
    for key, val in depth_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No depth metrics (GT depth not available)")

print("\nPose Metrics:")
if pose_metrics:
    for key, val in pose_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No pose metrics (GT poses not available)")


Resizing input from 256x320 to 224x280
Depth Metrics:
  abs_rel: 0.146308
  sq_rel: 1.799598
  rmse: 10.566817
  rmse_log: 0.193997
  a1: 0.755919
  a2: 0.980715
  a3: 0.999690
  median_scaling_ratio: 476.283569
  median_scaling_std: 0.047820

Pose Metrics:
  pose_trans_err_ang_deg: 87.982376
  pose_trans_err_scale: 0.310488
  pose_rot_err_deg: 0.186768
  pose_pred_rel_trans_scale: 0.000148
  pose_pred_f0_depth_scale: 0.089459


## 4. Test Training Step (Forward + Backward)


In [6]:
# Full training step
trainer.set_train()
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)

# Forward
outputs, losses = trainer.process_batch(inputs)

# Backward
trainer.model_optimizer.zero_grad()
losses["loss"].backward()
trainer.model_optimizer.step()

print(f"Training step completed! Loss: {losses['loss'].item():.6f}")


Resizing input from 256x320 to 224x280
Training step completed! Loss: 0.175582
